# prep_labse

Preprocesamiento (no es un experimento: solo genera datos).

Embedding de cada `Description` **en su idioma original** con LaBSE (Google, 109 idiomas, 768 dimensiones),
versión para `transformers` `setu4993/LaBSE`: se usa `pooler_output` (CLS + capa densa) normalizado a largo 1,
que es la forma recomendada por el modelo. Local, sin servicios externos. No usa AdoptionSpeed.

Salida: `train.parquet` y `test.parquet` con índice `PetID` y columnas `lbs_0..lbs_767`
(vacías si la mascota no tiene descripción).

In [1]:
import os, glob, time
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizerFast, BertModel

torch.manual_seed(42)
MODELO = "setu4993/LaBSE"
tok = BertTokenizerFast.from_pretrained(MODELO)
mod = BertModel.from_pretrained(MODELO).eval()

def embeber(textos, lote=32):
    salida = []
    for k in range(0, len(textos), lote):
        enc = tok(textos[k:k + lote], padding=True, truncation=True, max_length=256, return_tensors="pt")
        with torch.no_grad():
            v = mod(**enc).pooler_output
        salida.append(torch.nn.functional.normalize(v, dim=1).numpy())
    return np.vstack(salida)

C:\Users\eoinb\projects\labo2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   9%|▉         | 18/199 [00:00<00:01, 169.39it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1005.36it/s]

In [2]:
RAIZ = "/kaggle/input" if os.path.exists("/kaggle/input") else "../input"
COMP = os.path.dirname(os.path.dirname(glob.glob(f"{RAIZ}/**/train/train.csv", recursive=True)[0]))
SALIDA = "." if os.path.exists("/kaggle") else "../input/lab2-labse"
os.makedirs(SALIDA, exist_ok=True)
for conjunto in ["train", "test"]:
    t0 = time.time()
    df = pd.read_csv(f"{COMP}/{conjunto}/{conjunto}.csv", index_col="PetID", usecols=["PetID", "Description"])
    tiene = df["Description"].notna() & (df["Description"].astype(str).str.strip() != "")
    E = embeber(df.loc[tiene, "Description"].astype(str).tolist())
    res = pd.DataFrame(np.nan, index=df.index, columns=[f"lbs_{j}" for j in range(E.shape[1])], dtype="float32")
    res.loc[tiene] = E.astype("float32")
    res.to_parquet(os.path.join(SALIDA, f"{conjunto}.parquet"))
    print(conjunto, res.shape, "| con descripción:", int(tiene.sum()), f"| {(time.time() - t0) / 60:.1f} min")

train (14993, 768) | con descripción: 14980 | 42.7 min


test (3972, 768) | con descripción: 3971 | 11.8 min


In [3]:
# Control: frases equivalentes en distintos idiomas deberían quedar cerca (coseno alto)
v = embeber(["very cute kitten, free for adoption", "anak kucing sangat comel, percuma untuk diadopsi",
             "非常可爱的小猫，免费领养", "the car needs new tyres"])
print(np.round(v @ v.T, 2))

[[1.   0.88 0.88 0.22]
 [0.88 1.   0.8  0.37]
 [0.88 0.8  1.   0.18]
 [0.22 0.37 0.18 1.  ]]
